# Lab 4 — Plug a Ray Data Pipeline into Ray Train (AI-assisted)

**Time:** ~15–20 min  
**Mode:** AI-assisted.

## Where this picks up
Lab 3 ported a vanilla PyTorch loop to Ray Train. The lecture's `Training` notebook
then integrates Ray Data: it loads `synthetic_ecom_users_1000.ndjson`, looks up a
stable integer **user index** (0..N-1) for every record, explodes the
`last_20_positive_item_interactions` list column into (user, item) pairs, and
feeds the result to the trainer via `ray.train.get_dataset_shard`.

Your job: get an AI assistant to help you wire the Ray Data → Ray Train glue.

## Learning objectives
1. Chain a stateful actor stage (`MiniDatabaseFacade` — provided) into a Ray Data
   pipeline using `map_batches` with `fn_constructor_args`.
2. Write an `explode_interactions` step that flattens to (user_idx, item_idx) pairs.
3. Pass the pipeline to `TorchTrainer` via `datasets={'train': ds}` and consume a
   *shard* per worker with `iter_torch_batches`.
4. Recognize the difference between streaming and `materialize()`d datasets.

## What you do **not** have to invent
Translating a string user `id` into a stable integer index in `[0, num_users)` is
fiddly in a distributed pipeline (row position within a shard is not the same as
row position in the full dataset). The setup cell below provides a `MiniDatabaseFacade`
class — modeled on the lecture's `DatabaseFacade` — that does the lookup for you.
Use it as-is.

## Setup — model, paths, and the provided DatabaseFacade

In [ ]:
import ray, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, numpy as np
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig
import ray.train as ray_train

if not ray.is_initialized():
    ray.init()

USERS_PATH = '/mnt/cluster_storage/ecom/users.ndjson'
NUM_USERS, NUM_ITEMS, DIM = 1000, 1000, 64

class TwoTower(nn.Module):
    def __init__(self, n_u, n_i, dim):
        super().__init__()
        self.u = nn.Embedding(n_u, dim)
        self.v = nn.Embedding(n_i, dim)
    def forward(self, u, i):
        return F.normalize(self.u(u), dim=-1) @ F.normalize(self.v(i), dim=-1).t()

In [ ]:
# Provided for you — same shape as the lecture's DatabaseFacade.
# Loads the user table once per actor replica, then maps a batch of string user `id`s
# to their integer row positions in the original users file. Those row positions
# ARE the stable user indices in [0, NUM_USERS) that the TwoTower model expects.
class MiniDatabaseFacade:
    def __init__(self, users_path: str):
        self.users = pd.read_json(users_path, lines=True)

    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]

    def __call__(self, batch):
        # Add a `user_indices` column to each batch.
        # NOTE: assumes batch['id'] order matches the lookup order — true here
        # because the underlying DataFrame preserves the original file order.
        batch['user_indices'] = self.users_for_ids(batch['id']).index.values
        return batch

## Inspect the source data
Confirm the columns we'll work with.

In [ ]:
ds = ray.data.read_json(USERS_PATH, lines=True, file_extensions=['.ndjson'])
print('count:', ds.count())
ds.select_columns(['id', 'last_20_positive_item_interactions']).take(1)

## Exercise — drive the AI assistant

Tell the assistant up front that **`MiniDatabaseFacade` is already defined** —
you don't want it re-implementing the user-id lookup or inventing a row-position
workaround. Have it use `MiniDatabaseFacade` via `map_batches` with
`fn_constructor_args=[USERS_PATH]`.

### Prompt A — build the preprocessing pipeline

> *"Using Ray Data, build a preprocessing pipeline starting from `ds`. It must:
> 1. Select columns `id` and `last_20_positive_item_interactions`.
> 2. Use the **already-provided** `MiniDatabaseFacade` as a `map_batches` actor stage
>    (`fn_constructor_args=[USERS_PATH]`) so each batch gains a `user_indices` column.
>    Do not write your own user-id lookup or use row position as a stand-in.
> 3. Add a second `map_batches` step `explode_interactions(batch)` that turns each
>    user's 20 interactions into 20 pairs. The output schema must be exactly
>    `{'user_indices': np.ndarray, 'interactions': np.ndarray}` with the two arrays
>    aligned 1:1.
> 4. Limit to the first 10,000 *output rows* for fast iteration."*

**Acceptance criteria:**
- `processed.take(2)` shows two records each with `user_indices` and `interactions`
  as ints (not lists).
- Every `user_indices` value is in `[0, NUM_USERS)`.
- For any single source user, all of that user's emitted rows share the same
  `user_indices` value (i.e. the lookup result is consistent).

In [29]:
# add AI assisted/generated code

### Prompt B — feed it to `TorchTrainer`

Create a prompt that includes the `train_loop_ray` code from Lab 4.1 as follows:

> *"Adapt `train_loop_ray` to consume the dataset via
> `ray.train.get_dataset_shard('train').iter_torch_batches(batch_size=128)`.
> Each batch is a dict with keys `user_indices` and `interactions` — pass them as
> the user/item ids into the TwoTower model. Report the running mean loss per
> epoch with `ray.train.report`. Wire everything together with
> `TorchTrainer(..., datasets={'train': processed})` and `ScalingConfig(num_workers=2)`."*

Verify:
1. `iter_torch_batches` is called inside the worker function — not in the driver.
2. Both `user_indices` and `interactions` are cast to `.long()` before the model call.
3. Training completes; metrics appear in the logs.

In [30]:
# add AI assisted/generated code

### Prompt C — streaming vs. materialize

Run the training twice: once with the streaming dataset, once after calling `processed.materialize()` first.

Provide the logging output of each of the two runs to the AI and ask it to compare the Ray Data logging output for both runs and explain the difference in 2 sentences.

## Wrap-up
- The `datasets={...}` argument splits the dataset into shards — one per worker.
- `iter_torch_batches` is the streaming bridge between Ray Data and PyTorch.
- A *stateful* `map_batches` actor (like `MiniDatabaseFacade`) is the right tool
  any time you have a heavy resource (a model, a database, a lookup table) that
  should load **once per replica** and serve many batches.
- `materialize()` writes the pipeline output into the object store, so multi-epoch
  training reads from cache instead of re-running the pipeline.